### Why this exists

A chunk store answers "which 400 characters mention this". That is the wrong unit for half of the
questions people actually ask a corpus:

| question | what the chunk store gives you | what it should give you |
|---|---|---|
| "where in this 500-page book is X discussed?" | five fragments | a chapter |
| "summarise section 3" | fragments that mention section 3 | section 3 |
| "fill this template from these reports" | fragments | cited sections per question |

The fix is not a better chunker. It is remembering **where each chunk lives**. `litesearch.tree`
builds a [PageIndex](https://github.com/VectifyAI/PageIndex)-style node tree per document at
ingest time, links every chunk to a node, and adds three things on top of the existing hybrid
search: evidence that **rolls up** to sections, `toc()`/`read()` for reasoning over structure with
no embeddings at all, and chunk **spans** so a generating model sees contiguous text.

Nothing here calls a language model. PageIndex builds its tree with one; the structural signal —
markdown headings, then chapter lines, then page windows as the floor — gets most of the value at
zero token cost, and every hook that *would* want a model (`summarize=`, `build=`) takes a
callable, so one can be dropped in exactly where it pays.

> Adapted from `chitragupta`, which prototyped this over an astrology-book library. What lives here
> is the part that has nothing to do with any particular corpus.

In [ ]:
#| default_exp tree

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.all import Path, patch, first, L, chunked, AttrDict, ifnone, defaults, parallel
from fastlite import Database
from apswutils.db import Table
from apswutils.utils import hash_record
from dataclasses import dataclass, field
import re, tempfile
import numpy as np

from litesearch.core import _in, _rid, _np_dtype, rrf_merge, process_content, write_txn, db_lock
from litesearch.data import chunk_markdown

## Detecting structure

Three signals, tried in order. Markdown headings are the strongest and the most common (anything
that went through `pdf_markdown`, a docs site, or a notebook has them). Classic books that were
scanned to plain text have chapter lines instead. Everything else falls back to fixed page
windows, which are a poor tree but keep the invariant that *every* document is navigable — a
`toc()` that sometimes returns nothing is a `toc()` nobody calls.

#### `struct_levels`

`{heading word: level}` for one document, as consecutive levels starting at 1.

Levels are compacted rather than fixed, so a directive that only ever says "Article" gets
articles at level 1 instead of burying every one of them four deep. Unknown words are ranked
by frequency: the rarer heading word names the bigger division.

#### `detect_mode`

Which structural signal this document carries: `markdown`, `verse`, `chapter` or `window`.

`verse` is tried before the markdown/chapter pair because a Sanskrit source satisfies neither
of them and would fall all the way through to `window` — measured on GRETIL's Manusmṛti, all
twelve adhyāyas and 2,685 verses collapsing into a single `Pages 1–1` node. Its own signal is
the citation marker, which is unambiguous and carries the hierarchy for free.

In [ ]:
#| export
_md_head  = re.compile(r'^(#{1,6})\s+(.+?)\s*#*\s*$')
_fence    = re.compile(r'^\s{0,3}(```+|~~~+)')
# A structural heading line, with an optional leading `#` — a PDF converter may already have
# marked it up, and the word is a better signal than the markup either way.
_STRUCT = ('BOOK','TITLE','PART','ANNEX','APPENDIX','CANTO','CHAPTER','ADHYAYA','SECTION','SUBTITLE',
           'ARTICLE','RULE','CLAUSE','LESSON','SCHEDULE')
_chapter  = re.compile(r'^\s{0,6}#{0,6}\s*(' + '|'.join(_STRUCT) + r')\s+([IVXLCDM]+|\d+[A-Za-z]?)'
                       r'\b[\s.:—-]*(.{0,80})$', re.I)
# Prior order of the usual hierarchy words. Lower sits higher in the tree. Anything not listed is
# ranked by how often it occurs, since a rarer heading word names a bigger division.
_STRUCT_RANK = {'book':0, 'title':0, 'part':0, 'annex':0, 'appendix':0, 'canto':0,
                'chapter':1, 'adhyaya':1, 'section':2, 'subtitle':2,
                'article':3, 'rule':3, 'clause':3, 'lesson':3, 'schedule':3}
# Above this many markdown headings per page, `#` has stopped meaning "heading" — see `detect_mode`.
MAX_HEAD_DENSITY = 4.0
_ws = re.compile(r'\s+')

def _md_lines(txt):
    "Yield `(line, in_code)` for `txt`, A fenced block's lines are kept as text, but never read as a heading."
    fence = None
    for ln in (txt or '').splitlines():
        if (m := _fence.match(ln)):
            tok = m.group(1)
            if fence is None: fence = tok
            elif tok[0] == fence[0] and len(tok) >= len(fence): fence = None
            yield ln, True
        else: yield ln, fence is not None

@dataclass
class TreeNode:
    'One section of a document. `seq` is its index in the flat list; `parent` is another `seq`.'
    seq:int
    title:str
    level:int
    parent:int|None = None
    page_start:int = 0
    page_end:int = 0
    segments:list = field(default_factory=list)   # [(page, text), ...]
    children:list = field(default_factory=list)   # [seq, ...]
    summary:str = ''
    def text(self): return '\n\n'.join(s for _, s in self.segments if s.strip())

def summarize_extractive(text:str, n:int=300) -> str:
    'First `n` clean characters. Cheap, deterministic, and the seam an LLM slots into via `summarize=`.'
    t = _ws.sub(' ', text).strip()
    return t[:n] + ('…' if len(t) > n else '')

def _clean_title(t:str, max_len:int=100): return _ws.sub(' ', t).strip(' #*_')[:max_len].strip()

def struct_levels(pages, max_levels:int=4) -> dict:
    '`{heading word: level}` for one document, as consecutive levels starting at 1.'
    seen = {}
    for _, txt in pages:
        for ln, code in _md_lines(txt):
            if code: continue
            if (m := _chapter.match(ln)): seen[m.group(1).lower()] = seen.get(m.group(1).lower(), 0) + 1
    if not seen: return {}
    unknown = sorted((n, w) for w, n in seen.items() if w not in _STRUCT_RANK)
    rank = {w: (_STRUCT_RANK[w], 0) for w in seen if w in _STRUCT_RANK}
    rank |= {w: (99, i) for i, (_, w) in enumerate(unknown)}
    order = sorted({rank[w] for w in seen})
    return {w: min(order.index(rank[w]) + 1, max_levels) for w in seen}

def _md_stats(pages):
    'Markdown heading count and how many distinct `#` depths appear, ignoring fenced code.'
    n, lvls = 0, set()
    for _, txt in pages:
        for ln, code in _md_lines(txt):
            if code: continue
            if (m := _md_head.match(ln)): n += 1; lvls.add(len(m.group(1)))
    return n, len(lvls)

def _cite_stats(pages):
    'How many blocks carry a verse citation, and the deepest hierarchy any of them names.'
    from litesearch.sanskrit import CITE_RE, cite_parts
    n, depth = 0, 0
    for _, txt in pages:
        if (m := CITE_RE.search(txt or '')):
            n += 1
            depth = max(depth, len(cite_parts(f'{m.group(1)}_{m.group(2)}')[1]))
    return n, depth

def detect_mode(pages) -> str:
    'Which structural signal this document carries: `markdown`, `verse`, `chapter` or `window`.'
    md, depths = _md_stats(pages)
    ch = sum(1 for _, txt in pages for ln, code in _md_lines(txt) if not code and _chapter.match(ln))
    npg = max(1, len(pages))
    nc, cdepth = _cite_stats(pages)
    # a citation on a real share of the blocks, naming more than a flat sequence
    if nc >= 3 and nc/npg > 0.2 and cdepth >= 2 and nc > md: return 'verse'
    if md/npg > MAX_HEAD_DENSITY and depths < 2: return 'chapter' if ch >= 3 else 'window'
    if md >= (2 if len(pages) <= 3 else max(3, npg // 25)): return 'markdown'
    return 'chapter' if ch >= 3 else 'window'


`build_tree` returns a **flat list** rather than a nested one: `nodes[seq]` is O(1), the parent
link is an int, and it maps onto a SQL table without a serialiser. Nesting is a view over it
(`toc()` builds one on demand).

A heading is the other way a Sanskrit source names its divisions: GRETIL addresses
verses by citation, but a stotra names sections (`## dhyanam`) and carries no
citation at all. Verse mode has to read both or one of them gets a flat tree.

Citation depth is *relative to the heading it sits under*. The two signals
count from different origins — `#` depth starts wherever the markup starts,
a citation's hierarchy always starts at 1 — so sharing one level space makes
`open_node` pop the heading off the stack and reparent its verses to the
root. TEI is exactly this case (`<div type="adhyāya">` *and*
`<lg xml:id="Manu_1.1">`), and it came out with the adhyāya nodes empty and
the second one adopted by the first chapter's citation node.

#### `_dedent_path`

Drop a segment that only repeats the one before it.

A document whose first heading *is* its title — `# Field Manual` in `Field Manual.md`, and
every PDF whose H1 is its filename — otherwise reads `Field Manual › Field Manual › Chapter 1`
in every breadcrumb, on every hit, and in the text embedded with every chunk.

#### `heading_path`

`Doc › Part › Chapter` for one node.

Embedded with the chunk and indexed for FTS, this is what stops a chunk reading as an
out-of-context fragment: "the effects are severe" means nothing until you know which chapter
said it.

In [ ]:
#| export
def build_tree(pages,                  # [(page_no, text)] — markdown or plain text
               title:str='Document',   # title of the root node
               summarize=None,         # callable(text)->str for node summaries (LLM goes here)
               window:int=8,           # pages per node in `window` mode
               min_level:int=1,        # floor for heading depth
               max_levels:int=4,       # deepest node level kept
               mode:str=None           # force a structural mode instead of detecting one
) -> list:
    'A flat list of `TreeNode` for a document (index = seq, root = 0).'
    pages = [(p, t or '') for p, t in pages]
    summarize = summarize or summarize_extractive
    mode, nodes = mode or detect_mode(pages), []
    slev = struct_levels(pages, max_levels) if mode == 'chapter' else {}
    def fresh(t, lvl, parent, page):
        nd = TreeNode(seq=len(nodes), title=_clean_title(t) or f'Section {len(nodes)}',
                      level=lvl, parent=parent, page_start=page, page_end=page)
        nodes.append(nd)
        if parent is not None: nodes[parent].children.append(nd.seq)
        return nd
    root = fresh(title, 0, None, pages[0][0] if pages else 0)
    stack, cur, buf, cur_page = [root], root, [], root.page_start
    def open_node(t, lvl, page):
        'Attach a heading under the nearest open ancestor shallower than it.'
        nonlocal cur
        lvl = min(lvl, max_levels)
        while len(stack) > 1 and stack[-1].level >= lvl: stack.pop()
        cur = fresh(t, lvl, stack[-1].seq, page)
        stack.append(cur)
    def flush(page):
        if (txt := '\n'.join(buf).strip()): cur.segments.append((page, txt))
        cur.page_end = max(cur.page_end, page)
        buf.clear()

    if mode == 'verse':
        # `Mn_1.1` is both the verse's id and its address: everything but the last component names
        # the division it sits in, so the tree comes out of the citations with no markup at all.
        from litesearch.sanskrit import CITE_RE, cite_parts
        last, head_lvl = [], 0
        for p, txt in pages:
            if (h := first(ln for ln, code in _md_lines(txt) if not code and _md_head.match(ln))):
                m = _md_head.match(h)
                flush(cur_page)
                head_lvl = max(min_level, len(m.group(1)))
                open_node(m.group(2), head_lvl, p)
                cur_page, last = p, []
                if (rest := txt.replace(h, '', 1).strip()): buf.append(rest)
                flush(p); cur_page = p
                continue
            sig, parts = '', []
            if (m := CITE_RE.search(txt)):
                sig, nums = cite_parts(f'{m.group(1)}_{m.group(2)}')
                parts = nums[:-1][:max_levels]           # drop the verse itself; keep its address
            for lvl in range(1, len(parts)+1):
                if last[:lvl] != parts[:lvl]:
                    flush(cur_page)
                    open_node(f"{sig} {'.'.join(parts[:lvl])}".strip(), head_lvl + lvl, p)
                    cur_page = p
            last = parts
            if txt.strip(): buf.append(txt)
            flush(p)
            cur_page = p
    elif mode == 'window':
        for i in range(0, len(pages), window):
            grp = pages[i:i+window]
            lead = next((l.strip() for _, t in grp for l in t.splitlines() if l.strip()), '')
            open_node(f'Pages {grp[0][0]+1}–{grp[-1][0]+1}: {lead[:60]}', 1, grp[0][0])
            for p, t in grp:
                if t.strip(): cur.segments.append((p, t.strip()))
                cur.page_end = p
        cur = root
    else:
        for p, txt in pages:
            for ln, code in _md_lines(txt):
                m = None if code else (_md_head.match(ln) if mode == 'markdown' else _chapter.match(ln))
                # a lowercase first character usually means a `#` that was punctuation, not a heading
                if m and (mode == 'chapter' or (len(t := _clean_title(m.group(2))) >= 2 and not t[0].islower())):
                    flush(cur_page)
                    open_node(ln.strip() if mode == 'chapter' else m.group(2),
                              slev.get(m.group(1).lower(), 1) if mode == 'chapter'
                              else max(min_level, len(m.group(1))), p)
                    cur_page = p
                    continue
                buf.append(ln)
            flush(p)
            cur_page = p
    flush(cur_page)
    for nd in nodes:
        base = nd.text() or ' / '.join(nodes[c].title for c in nd.children[:8])
        nd.summary = summarize(base) if base else ''
    root.page_end = max((n.page_end for n in nodes), default=root.page_end)
    return nodes

def _dedent_path(parts) -> list:
    'Drop a segment that only repeats the one before it.'
    out = []
    for p in parts:
        if p and (not out or _ws.sub(' ', p).strip().lower() != _ws.sub(' ', out[-1]).strip().lower()):
            out.append(p)
    return out

def heading_path(tree,              # the node list from build_tree
                 nd,                # the TreeNode to describe
                 title:str,         # document title
                 sep:str=' › ',
                 max_len:int=200) -> str:
    '`Doc › Part › Chapter` for one node.'
    parts, cur = [], nd
    while cur is not None:
        if cur.level > 0: parts.append(cur.title)
        cur = tree[cur.parent] if cur.parent is not None else None
    return sep.join(_dedent_path([title] + parts[::-1]))[:max_len]


In [ ]:
tree = build_tree([(0, '# Saturn\n\nIntro text about the ringed planet.\n\n## Transits\n\nSade sati runs seven years.'),
                   (1, '## Remedies\n\nRecite on Saturdays.')], title='Jyotisha')
for n in tree: print(f'{n.seq}  lvl{n.level}  p{n.page_start}-{n.page_end}  {n.title!r:24} {n.summary[:40]!r}')
assert [n.title for n in tree] == ['Jyotisha', 'Saturn', 'Transits', 'Remedies']
assert tree[2].parent == 1 and tree[3].parent == 1, 'h2s hang off the h1, not off each other'
assert heading_path(tree, tree[2], 'Jyotisha') == 'Jyotisha › Saturn › Transits'

In [ ]:
# a skipped heading level must not nest siblings under each other. Both of these are ordinary
# documents: a package README whose h1 is lowercase (so the heading heuristic drops it) and leaves
# `##` as the shallowest heading, and a document that jumps from `#` straight to `###`.
_gap = build_tree([(0, '# litesearch\n\n> a tagline\n\n## Install\n\npip install\n\n'
                      '## Encoders\n\nencoders\n\n### Fallback\n\nhashing')], title='litesearch')
assert [n.title for n in _gap] == ['litesearch', 'Install', 'Encoders', 'Fallback']
assert [n.parent for n in _gap] == [None, 0, 0, 2], [n.parent for n in _gap]
assert heading_path(_gap, _gap[2], 'litesearch') == 'litesearch › Encoders'
assert heading_path(_gap, _gap[3], 'litesearch') == 'litesearch › Encoders › Fallback'

_jump = build_tree([(0, '# Doc\n\ntop\n\n### Deep One\n\na\n\n### Deep Two\n\nb')], title='Doc')
assert [n.parent for n in _jump] == [None, 0, 1, 1], [(n.title, n.level, n.parent) for n in _jump]

In [ ]:
#| hide
# `verse` mode reads two structural signals, and they count from different origins: `#` depth
# starts wherever the markup starts, a citation's hierarchy always starts at 1. Sharing one level
# space made `open_node` pop the heading off the stack — TEI (a `<div type="adhyāya">` *and* an
# `<lg xml:id="Manu_1.1">`) came out with empty adhyāya nodes and the second one adopted by the
# first chapter's citation node.
_mix = build_tree([(0,'## adhyāya 1'),(1,'v1 || Mn_1.1 ||'),(2,'v2 || Mn_1.2 ||'),
                   (3,'## adhyāya 2'),(4,'v3 || Mn_2.1 ||')], title='Manu', mode='verse')
_by = {n.title: n for n in _mix}
assert _by['Mn 1'].parent == _by['adhyāya 1'].seq, [(n.title, n.parent) for n in _mix]
assert _by['Mn 2'].parent == _by['adhyāya 2'].seq
assert _by['adhyāya 2'].parent == 0                      # a sibling of adhyāya 1, not its nephew
assert _by['Mn 1'].segments and _by['Mn 2'].segments     # and the verses landed somewhere
assert heading_path(_mix, _by['Mn 1'], 'Manu') == 'Manu › adhyāya 1 › Mn 1'

# the two single-signal shapes must be untouched by that: citation-only starts at level 1 ...
_cit = build_tree([(i, f'v || Mn_{c}.{v} ||') for i,(c,v) in enumerate([(1,1),(1,2),(2,1)])],
                  title='Manu', mode='verse')
assert [n.level for n in _cit] == [0,1,1], [n.level for n in _cit]
# ... and heading-only keeps its `#` depth
_hd = build_tree([(0,'## dhyanam'),(1,'text ॥'),(2,'## nyasam'),(3,'more ॥')],
                 title='Stotra', mode='verse')
assert [n.level for n in _hd] == [0,2,2] and all(n.parent in (None,0) for n in _hd)

# a document whose H1 *is* its title must not say so twice, in every breadcrumb of every chunk
_dup = build_tree([(0, '# Field Manual\n\nintro\n\n## Chapter 1\n\nbody')], title='Field Manual')
assert heading_path(_dup, _dup[-1], 'Field Manual') == 'Field Manual › Chapter 1', \
       heading_path(_dup, _dup[-1], 'Field Manual')
assert _dedent_path(['A','a ','A','B']) == ['A','B']       # case- and space-insensitive


In [ ]:
# a `#` comment inside a fenced block is a comment, not an h1 — otherwise a README's usage examples
# invent sections that adopt every real heading after them
_readme = '''# vishalakshi

> a tagline

## Harvest

```python
v.apis('https://shop/browse')
# [0] .../api/bff/products?page=1   records: 24
```

## Watches

watch what changes
'''
_fen = build_tree([(0, _readme)], title='vishalakshi')
assert [n.title for n in _fen] == ['vishalakshi', 'Harvest', 'Watches'], [n.title for n in _fen]
assert [n.parent for n in _fen] == [None, 0, 0]
assert 'records: 24' in _fen[1].text(), 'the fenced lines are still text, only never headings'
assert _md_stats([(0, _readme)]) == (3, 2)        # h1 + 2 h2; the comment is not a heading
assert detect_mode([(0, _readme)]) == 'markdown'

# an unclosed fence swallows the rest of the document rather than mis-reading it as structure
_open = build_tree([(0, '# Doc\n\n## Alpha\n\n```\n## Beta\n')], title='Doc')
assert [n.title for n in _open] == ['Doc', 'Doc', 'Alpha'], [n.title for n in _open]
assert '## Beta' in _open[2].text()

# a ````-fenced block quotes ``` blocks inside itself: only a fence at least as long closes it
_nest = '# Doc\n\n## Alpha\n\n````md\n```\n# Not a heading\n```\n````\n\n## Beta\n\nb'
assert [n.title for n in build_tree([(0, _nest)], title='Doc')] == ['Doc', 'Doc', 'Alpha', 'Beta']
# and the two fence characters do not close each other
assert list(_md_lines('~~~\n# x\n```\n# y\n~~~')) == [
    ('~~~', True), ('# x', True), ('```', True), ('# y', True), ('~~~', True)]

In [ ]:
# a scanned book has chapter lines, not markdown; and a doc with neither still gets a tree
_book = build_tree([(i, f'CHAPTER {i+1}\n\nbody of chapter {i+1}') for i in range(4)], title='Saravali')
assert detect_mode([(i, f'CHAPTER {i+1}\n\nbody') for i in range(4)]) == 'chapter'
assert len(_book) == 5 and _book[1].title.startswith('CHAPTER 1')

_plain = build_tree([(i, f'page {i} of undifferentiated prose') for i in range(20)], title='Notes', window=8)
assert detect_mode([(i, 'prose') for i in range(20)]) == 'window'
assert len(_plain) == 4, [n.title for n in _plain]   # root + ceil(20/8) windows
assert _plain[1].title.startswith('Pages 1–8')

## The tables

`get_tree` sits beside `get_store` the way `get_graph` does: two extra tables and three extra
columns on the chunk store. The store stays the source of truth and everything else in litesearch
— FTS5, the ANN index, `db.search`, the graph layer — keeps working unchanged on it.

```
docs    id · title · source · kind · pages · meta · added_at
nodes   id ('doc#seq') · doc_id · parent_id · level · seq · title ·
        page_start · page_end · summary · nchunks
store   content · embedding · metadata · doc_id · node_id · page · heading   [+ FTS5, +ANN]
```

Doc ids are content-addressed on `(source, title)`, so re-adding a document is a no-op rather than
a duplicate — the same property `hash=True` gives chunks.

In [ ]:
#| export
def doc_id(source, title='') -> str:
    'Content-addressed document id — re-adding the same source is a no-op, not a duplicate.'
    return hash_record({'k': f'{source}|{title}'})[:16]

@patch
def get_tree(self:Database,
             store:str='store',   # chunk store the tree is built over
             prefix:str=None,     # table prefix (default: '' for 'store', else '<store>_')
             ann:bool=True,       # register an ANN index on the chunk store
             **kw                 # extra typed columns for the chunk store
) -> AttrDict:
    'Create the docs/nodes tables and a node-aware chunk store. Idempotent; returns the tables.'
    p = prefix if prefix is not None else ('' if store == 'store' else f'{store}_')
    dt, nt = f'{p}docs', f'{p}nodes'
    st = self.get_store(store, hash=True, ann=ann, doc_id=str, node_id=str, page=int, heading=str, **kw)
    self.t[dt].create(id=str, title=str, source=str, kind=str, pages=int, meta=str, added_at=float,
                      pk='id', if_not_exists=True, defaults=dict(added_at='CURRENT_TIMESTAMP'))
    self.t[nt].create(id=str, doc_id=str, parent_id=str, level=int, seq=int, title=str,
                      page_start=int, page_end=int, summary=str, nchunks=int, pk='id', if_not_exists=True)
    for t, c in ((nt,'doc_id'), (nt,'parent_id'), (store,'doc_id'), (store,'node_id')):
        self.t[t].create_index([c], if_not_exists=True)
    return AttrDict(docs=self.t[dt], nodes=self.t[nt], store=st, prefix=p)

## Ingestion

`add_doc` is the whole pipeline: tree → node-aware chunks → embed → store. It takes `pages` rather
than a file, so acquisition stays someone else's problem — `litesearch.data.pdf_parse` for PDFs,
`file_parse` for anything on disk, fossick for the web. The one thing it insists on is that a
chunk carries its `heading` path, both embedded with the text and stored for FTS.

#### `_pack_target`

The size a chunker wants to pack *across* segments to, or 0 for the usual one-call-per-segment.

A chunker only ever sees one segment at a time, because a segment is what carries a page number.
That is invisible until a reader hands the tree one segment per unit: GRETIL prints one verse
per line and `gretil_parse` makes each verse its own page, so `ProseChunker`'s 700-character
target was being applied to a 60-character verse and could never reach — the Īśopaniṣad came out
at 10 chunks through either chunker, and the larger budget was decorative. Opting in through an
attribute keeps the chunker interface as it was for everything that does not care.

#### `_node_chunks`

Chunk every node segment, tagging each chunk with its node, page and heading path.

A chunk under `min_chunk` is *merged into its predecessor*, never dropped. Dropping is the
tempting reading of a minimum size and it is wrong here: `read()` assembles a section out of
its chunks, so a discarded chunk is text that has silently left the corpus.

`meta_fn(text) -> dict` annotates each chunk with facets only its format knows — a `Profile`
supplies it, and `litesearch.sanskrit` uses it for metre. It runs **after** the short chunks
have been merged, so a chunk's metadata always describes the text it actually ended up holding
rather than the fragment it started as; annotating first and merging after is the version of
this that quietly files a verse under the metre of its neighbour.

#### `add_doc`

Ingest one document: build its tree, chunk it per node, embed and store.

Chunks are embedded as `heading ⏎⏎ content` but **stored** as bare content, so retrieval sees
the context and the caller gets clean text back.

In [ ]:
#| export
MIN_CHUNK = 40

def _pack_target(chunker) -> int:
    'The size a chunker wants to pack *across* segments to, or 0 for the usual one-call-per-segment.'
    return int(getattr(chunker, 'target', 0) or 0) if getattr(chunker, 'pack_cited', False) else 0

def _pack_segments(rows, target:int) -> L:
    'Join consecutive chunks of one node up to `target` characters; the span keeps its first page.'
    if not target or len(rows) < 2: return L(rows)
    out = L()
    for r in rows:
        if out and len(out[-1]['content']) + len(r['content']) + 2 <= target:
            out[-1]['content'] = f"{out[-1]['content']}\n\n{r['content']}"
            continue
        out.append(r)
    return out

def _node_chunks(tree, title, did, chunker=None, min_chunk=MIN_CHUNK, meta_fn=None):
    'Chunk every node segment, tagging each chunk with its node, page and heading path.'
    out = L()
    for nd in tree:
        head, nid = heading_path(tree, nd, title), f'{did}#{nd.seq}'
        node_out = L()
        for page, seg in nd.segments:
            prev = None
            for c in chunk_markdown(seg, chunker):
                if not (c or '').strip(): continue
                if prev is not None and len(c.strip()) < min_chunk:
                    prev['content'] = f"{prev['content']}\n\n{c}"
                    continue
                prev = dict(content=c, doc_id=did, node_id=nid, page=page, heading=head)
                node_out.append(prev)
        out += _pack_segments(node_out, _pack_target(chunker))
    if meta_fn:
        import json as _json
        # written on every chunk, including as `{}` — heterogeneous keys across rows are what
        # `insert_all` cannot take, and a prose chunk having no facets is not a reason to omit it
        for c in out: c['metadata'] = _json.dumps(meta_fn(c['content']) or {}, ensure_ascii=False)
    return out

@patch
def _tree_ann(self:Database, store):
    '''Whether this store should carry an ANN index, without deciding it on the caller's behalf.

    `add_doc` calls `get_tree` internally, and `get_tree` defaults to `ann=True` — so a caller who
    opted out with `get_tree(store, ann=False)` had the index (and its per-document rebuild)
    registered underneath them on the first `add_doc`. A store that already exists keeps whatever
    it was created with; only a store being created here gets the default.'''
    return store not in self.t or bool(self._ann_meta(store))

@patch
def add_doc(self:Database,
            pages,                  # [(page_no, text)] — or a single string
            title:str,              # document title
            source:str=None,        # path or url (defaults to the title)
            kind:str='text',        # 'pdf' | 'web' | 'md' | 'code' | anything you filter on
            store:str='store',      # chunk store
            prefix:str=None,        # tree table prefix
            emb_fn=None,            # embedder: list[str] -> vectors
            chunker=None,           # chonkie chunker (default: FastChunker via chunk_markdown)
            summarize=None,         # callable(text)->str for node summaries
            with_heading:bool=True, # embed each chunk together with its heading path
            meta:dict=None,         # arbitrary json metadata for the doc row
            force:bool=False,       # re-ingest a document already present
            index:bool=True,        # mirror this document's chunks into the ANN index
            mode:str=None,          # force a build_tree structural mode instead of detecting one
            meta_fn=None,           # (chunk text) -> dict of facets stored as the chunk's `metadata`
            defer:list=None         # collect chunks here instead of embedding and storing them
) -> dict:
    '''Ingest one document: build its tree, chunk it per node, embed and store.'''
    import json
    with db_lock(self):
        if isinstance(pages, str): pages = [(0, pages)]
        pages = [(p, t or '') for p, t in pages]
        g, src = self.get_tree(store, prefix, ann=self._tree_ann(store)), str(ifnone(source, title))
        did = doc_id(src, title)
        if first(g.docs(where=f'id={did!r}')):
            if not force: return dict(doc_id=did, title=title, skipped='already ingested; pass force=True')
            self.delete_doc(did, store, prefix)
        tree = build_tree(pages, title=title, summarize=summarize, mode=mode)
        chunks = _node_chunks(tree, title, did, chunker, meta_fn=meta_fn)
        counts = {}
        for c in chunks: counts[c['node_id']] = counts.get(c['node_id'], 0) + 1
        with write_txn(self):
            g.docs.insert(dict(id=did, title=title, source=src, kind=kind,
                               pages=(max(p for p, _ in pages)+1 if pages else 0),
                               meta=json.dumps(meta or {})), replace=True)
            g.nodes.insert_all([dict(id=f'{did}#{nd.seq}', doc_id=did, seq=nd.seq, level=nd.level,
                                     parent_id=None if nd.parent is None else f'{did}#{nd.parent}',
                                     title=nd.title, page_start=nd.page_start, page_end=nd.page_end,
                                     summary=nd.summary, nchunks=counts.get(f'{did}#{nd.seq}', 0))
                                for nd in tree], replace=True)
        if chunks and defer is not None: defer.extend(chunks)
        elif chunks:
            store_chunks(g.store, chunks, emb_fn, with_heading)
            if index and (m := self._ann_meta(store)):
                ids = L(g.store(select='id', where=f'doc_id={did!r}')).itemgot('id')
                if ids: g.store._sync_index(list(ids), [], _np_dtype.get(m['dtype'], np.float16))
        return dict(doc_id=did, title=title, kind=kind, nodes=len(tree), chunks=len(chunks))

def store_chunks(store,                 # chunk store table
                 chunks,                # chunk dicts from `_node_chunks`, from one document or many
                 emb_fn=None,           # embedder: list[str] -> vectors
                 with_heading=True):    # embed each chunk together with its heading path
    '''Embed a batch of chunks and upsert them. The batch may span documents, which is the point.'''
    rows = [c for c in chunks if (c.get('content') or '').strip()]
    if not rows: return 0
    if emb_fn:
        txt = [f"{h}\n\n{c['content']}" if (with_heading and (h := c.get('heading'))) else c['content']
               for c in rows]
        for c, v in zip(rows, emb_fn(txt)): c['embedding'] = np.asarray(v).tobytes()
    process_content(store, rows, embed=False, hash_id_columns=['node_id','content'])
    return len(rows)

@patch
def delete_doc(self:Database, did:str, store:str='store', prefix:str=None):
    'Remove a document, its nodes and its chunks, dropping just those keys from the ANN index.'
    with db_lock(self):
        g = self.get_tree(store, prefix, ann=self._tree_ann(store))
        m = self._ann_meta(store)
        rm = L(g.store(select=_rid(), where=f'doc_id={did!r}')).itemgot('rowid') if m else L()
        with write_txn(self):
            g.store.delete_where(where=f'doc_id={did!r}')
            g.nodes.delete_where(where=f'doc_id={did!r}')
            g.docs.delete_where(where=f'id={did!r}')
        if m and rm: g.store._sync_index([], list(rm), _np_dtype.get(m['dtype'], np.float16))


In [ ]:
#| hide
# A chunker sees one segment at a time, so a reader that emits one verse per page defeats any
# packing target the chunker has. `_pack_segments` is the seam that lets a chunker ask for more.
_rows = [dict(content=f'verse {i} || X_{i} ||', doc_id='d', node_id='d#1', page=i, heading='h')
         for i in range(8)]
assert len(_pack_segments(_rows, 0)) == 8                     # off by default: nothing changes
# 40, not 200: each row is 17 characters, so a 200-character target swallows all eight and the
# assertion below could never see packing *stop*. The bound has to be small enough to bite.
_packed = _pack_segments([dict(r) for r in _rows], 40)
assert 1 < len(_packed) < 8, len(_packed)
assert all(len(p['content']) <= 40 for p in _packed)          # the target is a real bound
assert _packed[0]['page'] == 0                                # a span keeps its first page
# no text is lost or reordered by packing
assert ''.join(''.join(p['content'].split()) for p in _packed) == \
       ''.join(''.join(r['content'].split()) for r in _rows)

from litesearch.sanskrit import VerseChunker as _VC, ProseChunker as _PC
assert _pack_target(_VC()) == 0                               # verse profile: one chunk per citation
assert _pack_target(_PC()) == 700                             # prose profile: pack to its target
assert _pack_target(None) == 0


`add_file` is the one-liner for the common case. It only knows about *documents* — PDFs, markdown,
plain text, notebooks. Source code is a different tree (module › class › function) and belongs to
`kosha`, which builds it from the AST rather than from headings.

#### `add_file`

Ingest one document file. PDFs page through `pdf_parse`; everything else is one page of text.

A registered `Profile` wins over the extension table: it is the only thing that can tell a TEI
edition from any other `.xml`, and it carries the chunker, tree mode and per-chunk facets that
format needs.

`profile=` names one directly, which is the only way to reach a profile that cannot be
detected: `sanskrit_prose` shares every signal with `sanskrit_verse` and differs only in its
chunk budget, so nothing but the caller can tell them apart.

In [ ]:
#| export
DOC_EXTS = '.pdf,.md,.markdown,.txt,.rst,.ipynb,.xml,.tei,.htm,.html,.conllu'

@patch
def assets(self:Database, name:str=None) -> Path:
    'Where extracted assets (PDF images) go: beside the database file, never the working directory.'
    f = self.conn.filename
    d = (Path(f).parent if f else Path(tempfile.gettempdir())/'litesearch')/'assets'
    d.mkdir(parents=True, exist_ok=True)
    return d/name if name else d

@patch
def add_file(self:Database,
             path,                 # file to ingest
             title:str=None,       # defaults to a prettified filename
             store:str='store',
             prefix:str=None,
             kind:str=None,        # overrides the kind inferred from the extension
             out_path=None,        # dir for extracted PDF images; defaults to `assets/<stem>` beside the db
             profile:str=None,     # force a registered Profile by name instead of detecting one
             **kw                  # forwarded to add_doc (emb_fn, chunker, summarize, force, ...)
) -> dict:
    'Ingest one document file. PDFs page through `pdf_parse`; everything else is one page of text.'
    p = Path(path)
    parsed = _parse_doc((str(p), profile, str(out_path or self.assets(p.stem))))
    return self._add_parsed(p, parsed, title, store, prefix, kind, **kw)

def _parse_doc(args):
    '''Read one file to `(pages, meta, kind, profile)` — everything `add_doc` needs, and nothing
    that holds a database handle. Module level and plain-data in and out, so a process pool can
    pickle it; `add_dir` runs it on one when the parse is the expensive half.

    A profile is resolved *by name* here rather than re-detected, so a worker cannot silently pick
    a different reader from the parent. `None` back means this worker's registry does not have the
    profile the parent found — the caller re-parses that file itself rather than guessing.'''
    from litesearch.data import profile_for
    path, profile, assets = args
    p = Path(path)
    if (prof := profile_for(p, name=profile)) is not None and prof.parse:
        pages, pmeta = prof.parse(p)
        return list(pages), dict(pmeta or {}), (prof.kind or prof.name), prof.name
    if profile: return None                       # asked for a profile this process does not have
    if p.suffix.lower() == '.pdf':
        from litesearch.data import pdf_parse
        return list(enumerate(pdf_parse(str(p), out_path=assets))), {}, 'pdf', None
    if p.suffix.lower() == '.ipynb':
        from litesearch.data import ipynb_parse
        return [(0, '\n\n'.join(c['content'] for c in ipynb_parse(p)))], {}, 'notebook', None
    return [(0, p.read_text(errors='replace'))], {}, (p.suffix.lstrip('.').lower() or 'text'), None

@patch
def _add_parsed(self:Database, p, parsed, title=None, store='store', prefix=None, kind=None, **kw):
    'The half of `add_file` that touches the database, split out so the parse half can be pooled.'
    from litesearch.data import profile_for
    pages, pmeta, knd, pname = parsed
    ttl = title or Path(p).stem.replace('_',' ').replace('-',' ').strip()
    if pname and (prof := profile_for(name=pname)) is not None:
        # the chunker and the facet function are built in the parent: one is a live object and the
        # other a closure, and neither has any business crossing a process boundary
        if prof.chunker: kw.setdefault('chunker', prof.chunker())
        if prof.mode: kw.setdefault('mode', prof.mode)
        if prof.meta: kw.setdefault('meta_fn', prof.meta)
        return self.add_doc(pages, title or pmeta.get('title') or ttl, source=str(p),
                            kind=kind or knd, store=store, prefix=prefix,
                            meta=kw.pop('meta', None) or pmeta, **kw)
    return self.add_doc(pages, ttl, source=str(p), kind=kind or knd, store=store, prefix=prefix, **kw)

# Extensions whose *parse* is real work rather than a `read_text`. A pool only pays for these:
# reading 150 markdown files takes 2.7ms in total, which no amount of process startup improves.
PARSE_HEAVY = {'.pdf', '.xml', '.tei', '.htm', '.html', '.ipynb', '.conllu'}
# Below this many parse-heavy files, starting a pool costs more than the parse it splits.
MIN_PARALLEL_DOCS = 4

def _n_parse_workers(files, n_workers):
    'Resolve `n_workers`: None picks by how many files actually have an expensive parse.'
    if n_workers is not None: return n_workers
    heavy = sum(1 for p in files if p.suffix.lower() in PARSE_HEAVY)
    return 0 if heavy < MIN_PARALLEL_DOCS else defaults.cpus

A directory is a bulk load, so both indexes are built once for the walk rather than per file:
the FTS triggers are suspended by `bulk_load` and the ANN index is rebuilt at the end. Measured
over 400 markdown documents this is 16.0s against 112.2s, and the growth curve goes from
exponent 1.6 to 1.0 — which is the difference between a corpus that finishes and one that
does not.

Parsing runs on a process pool, writing does not, and the split is where it is because the two
halves are nothing alike. Over eight PDFs the parse is 3.55s of a 5.31s walk — two thirds, and
embarrassingly parallel. Over 150 markdown files the same "parse" is 2.7ms in total, so a pool
is pure loss and `n_workers=None` declines to start one: the choice is made on how many files
have a parse worth splitting, not on how many files there are. Writing stays in the parent
either way — there is one connection, one FTS index and one HNSW graph, and they are not
improved by being contended for.

Processes rather than threads: pdf-oxide does its work in Rust but never releases the GIL, so a
thread pool measured 0.76x on the equivalent code path in `data._parse_files`.

In [ ]:
#| export
@patch
def add_dir(self:Database,
            dir,                    # directory to walk
            types:str=DOC_EXTS,     # comma-separated extensions to ingest
            store:str='store',
            prefix:str=None,
            n_workers:int=None,     # parse workers; 0 is serial, None picks by parse-heavy file count
            embed_batch:int=2000,   # chunks embedded and written per flush; 0 writes per document
            **kw                    # forwarded to add_doc
) -> list:
    '''Ingest every document under a directory tree. Already-ingested sources are skipped, not duplicated.'''
    exts = {f".{t.strip().lstrip('.')}".lower() for t in types.split(',')}
    files = [p for p in sorted(Path(dir).rglob('*')) if p.is_file() and p.suffix.lower() in exts]
    g = self.get_tree(store, prefix, ann=self._tree_ann(store))
    prof, outp = kw.pop('profile', None), kw.pop('out_path', None)
    nw = _n_parse_workers(files, n_workers)
    pend, emb_fn, wh = [], kw.get('emb_fn'), kw.get('with_heading', True)
    def flush(force=False):
        if pend and (force or len(pend) >= embed_batch):
            store_chunks(g.store, pend, emb_fn, wh); pend.clear()
    if embed_batch: kw['defer'] = pend
    out = []
    with g.store.bulk_load():
        if nw and nw > 1 and files:
            jobs = [(str(p), prof, str(outp or self.assets(p.stem))) for p in files]
            for p, parsed in zip(files, parallel(_parse_doc, jobs, n_workers=nw, threadpool=False, progress=False)):
                if parsed is not None: out.append(self._add_parsed(p, parsed, store=store, prefix=prefix, index=False, **kw))
                else: out.append(self.add_file(p, store=store, prefix=prefix, index=False, profile=prof, out_path=outp, **kw))
                flush()
        else:
            for p in files:
                out.append(self.add_file(p, store=store, prefix=prefix, index=False, profile=prof, out_path=outp, **kw))
                flush()
        flush(force=True)
    if self._ann_meta(store): g.store.rebuild_index()
    return out

In [ ]:
#| hide
# The ANN index has to end up holding exactly the embedded chunks, however they arrived: in one
# `add_dir` bulk load, one at a time through `add_file`, or after a document is deleted again.
import tempfile, hashlib
from fastcore.test import test_eq
from litesearch.core import database
# a model-free embedder — sha256 digest read as 16 float16s, deterministic and offline
_e = lambda ts, **kw: np.stack([np.frombuffer(hashlib.sha256(t.encode()).digest(), dtype=np.float16) for t in ts])
_n = lambda t: first(_idb.q(f'select count(*) c from {t}'))['c']

_id = Path(tempfile.mkdtemp())
(_id/'a.md').write_text('# Alpha\n\n## One\n\nA passage about maritime vessel inspection duties and thorium decay.')
(_id/'b.md').write_text('# Beta\n\n## Two\n\nA second passage about polonium, radium and the isotopes between them.')
_idb = database(str(_id/'t.db'))
_idb.get_tree('store', ann=True)
_idb.add_dir(_id, emb_fn=_e)
test_eq(_idb.get_index('store').size, _n('store'))       # the bulk load leaves the index complete
test_eq(_n('store_fts'), _n('store'))                    # and the suspended FTS triggers rebuilt

(_id/'c.md').write_text('# Gamma\n\n## Three\n\nA third passage, added on its own rather than in a walk.')
_idb.add_file(_id/'c.md', emb_fn=_e)
test_eq(_idb.get_index('store').size, _n('store'))       # a single add_doc mirrors its own chunks
# the FTS triggers are live again after the bulk load, so this insert indexed on the way in
_walk = _idb.t.store.fts_search('walk', columns=['id'])
test_eq(len(_walk) > 0, True)

_did = first(d for d in _idb.toc() if d['title'] == 'c')['doc_id']
_idb.delete_doc(_did)
test_eq(_idb.get_index('store').size, _n('store'))       # and a delete drops just those keys


In [ ]:
#| hide
# `get_tree(ann=False)` has to survive an ingest. It used to be overwritten by add_doc's own
# `get_tree` call, which took the ann=True default and registered the index anyway.
_ndb = database()
_ndb.get_tree('store', ann=False)
_ndb.add_doc('# Plain\n\nA document ingested into a store that opted out of the ANN index.', title='plain')
test_eq(_ndb._ann_meta('store'), None)

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq
from litesearch import database
with tempfile.TemporaryDirectory() as td:
    td = Path(td).resolve()          # apsw reports the resolved path; /var is a symlink on macOS
    db = database(str(td/'t.db'))
    test_eq(db.assets(), td/'assets')
    test_eq(db.assets('paper'), td/'assets'/'paper')
    assert db.assets().exists()
    p = td/'notes.md'
    p.write_text('# Notes\n\nsome text about rank fusion\n')
    r = db.add_file(p)
    test_eq(r['kind'], 'md')                       # inferred from the extension
    r = db.add_file(p, title='Notes again', kind='note')
    test_eq(r['kind'], 'note')                     # and overridable
# an in-memory database has no directory of its own, so assets go somewhere writable
assert database(':memory:').assets().exists()


## Reading the structure

`toc` and `read` are the vectorless half — PageIndex's actual argument. An agent that already
knows it wants "the chapter on remedies" should not be asked to phrase that as a similarity query;
it should read the table of contents and open the node. No embedding is computed by either call.

In [ ]:
#| export
@patch
def toc(self:Database,
        doc:str=None,           # doc id, or a substring of the title (None = every document)
        store:str='store',
        prefix:str=None,
        max_depth:int=3,        # deepest level included
        summaries:bool=True     # include node summaries
) -> list:
    'The document tree as nested dicts — titles, page ranges, summaries. No embeddings touched.'
    g = self.get_tree(store, prefix)
    docs = list(g.docs(where=f'id={doc!r} or title like {"%"+doc+"%"!r}') if doc else g.docs())
    # every node for every document being listed, in one query and grouped here. One query per
    # document made `toc()` cost a round trip per document to build a listing whose whole point is
    # that it is the cheap, vectorless way to see the corpus.
    by_doc, ids = {}, [d['id'] for d in docs]
    for b in chunked(ids, 400):
        for r in g.nodes(where=_in('doc_id', b)): by_doc.setdefault(r['doc_id'], []).append(r)
    out = []
    for d in docs:
        rows = sorted(by_doc.get(d['id'], []), key=lambda r: r['seq'])
        kids = {}
        for r in rows: kids.setdefault(r['parent_id'], []).append(r)
        def node(r):
            o = dict(id=r['id'], title=r['title'], level=r['level'], nchunks=r['nchunks'],
                     pages=(r['page_start'], r['page_end']))
            if summaries and r['summary']: o['summary'] = r['summary']
            if r['level'] < max_depth and (ch := kids.get(r['id'])): o['children'] = [node(c) for c in ch]
            return o
        root = first(rows, lambda r: r['parent_id'] is None)
        out.append(dict(doc_id=d['id'], title=d['title'], source=d['source'], pages=d['pages'],
                        tree=node(root) if root else None))
    return out

@patch
def breadcrumb(self:Database, node_id:str, store:str='store', prefix:str=None, sep:str=' › ') -> str:
    'The path from the document down to a node: `Book › Chapter › Section`.'
    g = self.get_tree(store, prefix)
    parts, cur, seen = [], first(g.nodes(where=f'id={node_id!r}')), set()
    while cur and cur['id'] not in seen:
        seen.add(cur['id'])
        parts.append(cur['title'])
        cur = first(g.nodes(where=f'id={cur["parent_id"]!r}')) if cur['parent_id'] else None
    return sep.join(_dedent_path(parts[::-1]))

@patch
def read(self:Database,
         node_id:str,           # 'doc#seq', as returned by toc() or sections()
         store:str='store',
         prefix:str=None,
         max_chars:int=20000,   # cap on the assembled text
         children:bool=True     # append the text of child nodes too
) -> dict:
    'One whole section — the unit an agent should read instead of guessing from fragments.'
    g = self.get_tree(store, prefix)
    nd = first(g.nodes(where=f'id={node_id!r}'))
    if not nd: return {}
    ids = [node_id]
    if children:
        frontier = [node_id]
        while frontier:
            kids = [r['id'] for r in g.nodes(where=_in('parent_id', frontier))]
            if not kids: break
            ids += kids
            frontier = kids
    rows = sorted(g.store(select='content, node_id, page', where=_in('node_id', ids)),
                  key=lambda r: (ids.index(r['node_id']) if r['node_id'] in ids else 1<<30, r['page'] or 0))
    text = '\n\n'.join(r['content'] for r in rows)[:max_chars]
    return dict(id=node_id, title=nd['title'], breadcrumb=self.breadcrumb(node_id, store, prefix),
                pages=(nd['page_start'], nd['page_end']), summary=nd['summary'],
                children=[r['id'] for r in g.nodes(where=f'parent_id={node_id!r}')], text=text)

## Retrieval that knows about structure

Three refinements over `db.search`, each addressing a way a flat chunk list misleads:

**Adaptive fusion.** Plain RRF weights both legs equally on every query, which is wrong in two
directions that are cheap to detect. A quoted phrase or a name/number-heavy query ("Vimshottari",
`get_store`, "1706.03762") is a keyword query wearing a sentence's clothes; and when FTS returns
nothing at all, an equal vote just dilutes the vector ranking with noise. It is `adaptive=False`
by default: neither trigger fired once across 300 natural-language queries, so on prose it is
inert, and it is kept for the corpora where users really do paste quoted phrases.

**Spans.** Adjacent chunks that both hit are one passage cut in two. Merging them inside a node
gives a generating model contiguous text instead of two fragments with a hole in the middle.

**Section rollup.** `sections()` groups hits by node and scores each node by its **best** hit
(`score='max'`). Summing the RRF mass of every hit in a node is the intuitive alternative — five
weak hits spread across one chapter outranking one strong hit in an appendix — and it measures
badly, because it is a length prior in disguise: on 150 known-item queries over 486 pages of
legislation, summing cost 0.07 MRR against `max` on verbatim queries and 0.16 on keyword-degraded
ones, since long chapters outranked the precise article. `score='sum'` is still available for
corpora of uniform section length. Each result carries the `read()` handle for the next call,
which is the point: the agent's next action is in the payload.

#### `adaptive_weights`

`(fts_weight, vec_weight)` for RRF, from cheap signals in the query and the legs.

Three rules, not a learned model: a quoted phrase and a rare identifier are literal requests,
an empty leg cannot vote, and everything else is a tie. **Measured inert** on prose queries —
neither trigger fires on an ordinary sentence — so `doc_search(adaptive=...)` is off by
default. Worth turning on only where users really do type quoted phrases.

#### `merge_spans`

Collapse hits that are adjacent inside the same node into one span.

The merged hit keeps the best rank and score of its members, so ordering is unchanged; what
changes is that the caller gets one contiguous passage where it had two halves.

#### `doc_search`

Hybrid search over a node-aware store: span merging and a breadcrumb per hit.

`adaptive` defaults **off**. Its triggers — a quoted phrase, a name/number-heavy query — did
not fire once across 300 natural-language queries over 486 pages of legislation, giving
results identical to plain RRF to three decimal places, and a replacement rule that leaned on
the FTS leg's coverage measured *worse* (0.425 against 0.496 MRR on degraded queries). Kept
and opt-in: the case it was written for — users who really do type quoted phrases and
identifiers — is real, and simply is not what that corpus tests.

#### `sections`

Ranked *sections*, not chunks: hits grouped by node, each with a `read` handle.

`score='max'` — a section is as relevant as its best evidence. The obvious alternative, summing
the RRF mass of every hit in a node, reads well ("five weak hits in a chapter beat one strong
hit in an appendix") and measures badly: it is a length prior in disguise. On 150 known-item
queries over 486 pages of legislation, summing cost 0.07 MRR against `max` on verbatim queries
and 0.16 on keyword-degraded ones, because long chapters outranked the precise article. `mean`
is within noise of `max`; `sum` remains available for corpora of uniform section length.

In [ ]:
#| export
_QUOTED = re.compile(r'"[^"]+"')
_NAMEY  = re.compile(r'[A-Z][a-z]{2,}|\d|_')

def adaptive_weights(q:str,      # the raw query
                     fts:list,   # the FTS leg's results
                     vec:list    # the vector leg's results
) -> tuple:
    '`(fts_weight, vec_weight)` for RRF, from cheap signals in the query and the legs.'
    if not fts: return (0.0, 1.0)
    if not vec: return (1.0, 0.0)
    w = 1.0
    if _QUOTED.search(q or ''): w += 0.6
    toks = (q or '').split()
    if toks and sum(bool(_NAMEY.search(t)) for t in toks)/len(toks) >= 0.5: w += 0.4
    return (w, 1.0)

def merge_spans(hits,            # ranked hits carrying node_id + page
                gap:int=1        # merge hits at most this many pages apart
) -> list:
    'Collapse hits that are adjacent inside the same node into one span.'
    out, byn = [], {}
    for i, h in enumerate(hits):
        nid = h.get('node_id')
        if not nid: out.append(h); continue
        byn.setdefault(nid, []).append((i, h))
    for nid, group in byn.items():
        group.sort(key=lambda t: (t[1].get('page') or 0, t[0]))
        cur = None
        for i, h in group:
            if cur and (h.get('page') or 0) - (cur['_page_end'] or 0) <= gap:
                cur['content'] = f"{cur['content']}\n\n{h.get('content') or ''}"
                cur['_page_end'] = h.get('page') or cur['_page_end']
                cur['_nspan'] += 1
                cur['_rrf_score'] = max(cur.get('_rrf_score', 0), h.get('_rrf_score', 0))
                cur['_rank'] = min(cur['_rank'], i)
            else:
                cur = dict(h, _page_end=h.get('page'), _nspan=1, _rank=i)
                out.append(cur)
    return sorted(out, key=lambda h: h.get('_rank', 1<<30))

@patch
def doc_search(self:Database,
               q:str,                 # query string
               emb:bytes,             # query embedding
               columns:list=None,     # extra columns (node_id, page, heading, doc_id always included)
               limit:int=10,
               store:str='store',
               prefix:str=None,
               spans:bool=True,       # merge adjacent hits inside a node
               gap:int=1,             # span merge tolerance, in pages
               adaptive:bool=False,   # weight the RRF legs by query shape (measured inert — see below)
               rrf_k:int=60,
               **kw                   # forwarded to Database.search
) -> list:
    'Hybrid search over a node-aware store: span merging and a breadcrumb per hit.'
    cols = list(dict.fromkeys((columns or ['content']) + ['node_id','page','heading','doc_id','rowid']))
    base = self.search(q, emb, columns=cols, limit=max(limit*3, 30), table_name=store, rrf=False, **kw)
    if not base: return []
    fts, vec = base['fts'], base['vec']
    wf, wv = adaptive_weights(q, fts, vec) if adaptive else (1.0, 1.0)
    hits = _wrrf(fts, vec, wf, wv, rrf_k, limit*(3 if spans else 1))
    if spans: hits = merge_spans(hits, gap)
    for h in hits[:limit]: h['breadcrumb'] = h.get('heading') or self.breadcrumb(h.get('node_id') or '', store, prefix)
    return hits[:limit]

def _wrrf(fts, vec, wf=1.0, wv=1.0, k=60, limit=50, id_key='rowid'):
    'Weighted RRF over the two legs. `rrf_merge` with a thumb on the scale.'
    scores = {}
    for lst, w in ((fts, wf), (vec, wv)):
        for rank, row in enumerate(lst or []):
            rid = row.get(id_key, id(row))
            if rid in scores: scores[rid]['_rrf_score'] += w/(k+rank)
            else: scores[rid] = dict(row, _rrf_score=w/(k+rank))
    return sorted(scores.values(), key=lambda r: -r['_rrf_score'])[:limit]

@patch
def sections(self:Database,
             q:str,               # query string
             emb:bytes,           # query embedding
             limit:int=5,         # sections to return
             per:int=3,           # snippets kept per section
             store:str='store',
             prefix:str=None,
             fanout:int=8,        # chunk hits gathered before rolling up
             score:str='max',     # how a section scores from its hits: max | mean | sum
             **kw                 # forwarded to doc_search
) -> list:
    'Ranked *sections*, not chunks: hits grouped by node, each with a `read` handle.'
    hits = self.doc_search(q, emb, limit=max(limit*fanout, 20), store=store, prefix=prefix, spans=False, **kw)
    g, agg = self.get_tree(store, prefix), {}
    for h in hits:
        nid = h.get('node_id')
        if not nid: continue
        a = agg.setdefault(nid, dict(node_id=nid, score=0.0, _sum=0.0, _n=0, snippets=[], pages=[]))
        s_ = h.get('_rrf_score', 0.0)
        a['_sum'] += s_; a['_n'] += 1
        a['score'] = (a['_sum'] if score == 'sum' else
                      a['_sum']/a['_n'] if score == 'mean' else max(a['score'], s_))
        if len(a['snippets']) < per: a['snippets'].append((h.get('content') or '')[:400])
        if h.get('page') is not None: a['pages'].append(h['page'])
    if not agg: return []
    nodes = {r['id']: r for r in g.nodes(where=_in('id', list(agg)))}
    out = []
    for nid, a in sorted(agg.items(), key=lambda kv: -kv[1]['score'])[:limit]:
        nd = nodes.get(nid) or {}
        out.append(dict(node_id=nid, title=nd.get('title', ''), score=a['score'],
                        breadcrumb=self.breadcrumb(nid, store, prefix), summary=nd.get('summary', ''),
                        pages=(min(a['pages']) if a['pages'] else nd.get('page_start'),
                               max(a['pages']) if a['pages'] else nd.get('page_end')),
                        nchunks=nd.get('nchunks', 0), snippets=a['snippets'],
                        read=f'read({nid!r})'))
    return out


In [ ]:
#| export
def _prov(bc): return bool(bc) and '›' in bc   # a bare document title (the root) has no separator

@patch
def node_context(self:Database,
                 node_id:str,          # a 'doc#seq' node id
                 store:str='store',
                 prefix:str=None):
    'Where a section sits: its parent, siblings and children — a hit as a provision in a structure.'
    p = prefix if prefix is not None else ('' if store == 'store' else f'{store}_')
    N = self.t[f'{p}nodes']
    nd = first(N(where=f'id={node_id!r}'))
    if not nd: return AttrDict(parent=None, siblings=L(), children=L())
    pid = nd['parent_id']
    parent = first(N(where=f'id={pid!r}')) if pid else None
    sibs = L(N(where=f'parent_id={pid!r}')).filter(lambda r: r['id'] != node_id) if pid else L()
    kids = L(N(where=f'parent_id={node_id!r}'))
    pick = lambda r: AttrDict(id=r['id'], title=r['title'], level=r['level'])
    return AttrDict(parent=pick(parent) if parent else None,
                    siblings=sibs.sorted(key=lambda r: r['seq']).map(pick)[:8],
                    children=kids.sorted(key=lambda r: r['seq']).map(pick)[:12])

@patch
def context(self:Database,
            q:str,                  # query string
            emb:bytes,              # query embedding
            store:str='store',
            prefix:str=None,
            sections:int=6,         # operative sections returned
            per:int=3,              # snippets kept per section
            graph:bool=False,       # opt in to graph-reached related sections (see the note below)
            vector:bool=True,       # include embedding-nearest related sections
            related:int=8,          # max related sections
            max_read:int=6000,      # chars of assembled text per operative section
            tree_ctx:bool=True,     # attach each section's parent/siblings/children
            graph_w:float=0.6):
    '''One composed retrieval over a document tree: the operative sections plus what they connect to.

    `graph` is opt-in because the graph leg both helps and hurts, and which one depends on the
    query rather than on the corpus. Measured over three genres (`evals/multihop.py`,
    `evals/run.py eval_graph`):

    - On **known-item** queries — the target passage contains the words you searched for — it is a
      straight loss, monotonically worse as `graph_w` rises: p_mrr 0.8170 for plain hybrid against
      0.7395 / 0.6859 / 0.6463 at `graph_w` 0.25 / 0.5 / 1.0 on regulation, at 2-4x the latency.
    - On **bridge** queries — the target never uses the words you searched for and is relevant on
      structural grounds — it is a significant *win*, monotonically better as `graph_w` rises, in
      seven of nine paired-bootstrap comparisons (arXiv +0.0387 target MRR at `graph_w=1.0`,
      95% CI [+0.0110, +0.0694]).

    Most queries are known-item, so on as a default made most callers pay for a leg they do not
    use. Turn it on when you expect the corpus to answer by connection rather than by wording, and
    raise `graph_w` towards 1.0 when you do.'''
    p = prefix if prefix is not None else ('' if store == 'store' else f'{store}_')
    N, D, src = self.t[f'{p}nodes'], self.t[f'{p}docs'], {}
    def doc_of(nid):
        nd = first(N(where=f'id={nid!r}')); did = nd['doc_id'] if nd else None
        if did and did not in src:
            d = first(D(where=f'id={did!r}'))
            src[did] = Path(d['source']).name if (d and d['source']) else (d['title'] if d else did)
        return did, src.get(did)
    secs = self.sections(q, emb, limit=sections * 3, per=per, store=store, prefix=prefix)
    prim, seen, results = set(), set(), L()
    for s in secs:
        nid, bc = s.get('node_id'), s.get('breadcrumb')
        if not nid or not _prov(bc) or bc in seen: continue
        prim.add(nid); seen.add(bc)
        rd = self.read(nid, store=store, prefix=prefix, max_chars=max_read)
        did, fn = doc_of(nid)
        results.append(AttrDict(node_id=nid, doc_id=did, filename=fn,
            title=s.get('title') or rd.get('title'), breadcrumb=bc, pages=s.get('pages'),
            summary=rd.get('summary') or s.get('summary'), text=rd.get('text', ''),
            score=s.get('score', 0.0), snippets=L(s.get('snippets') or []),
            tree=self.node_context(nid, store, prefix) if tree_ctx else None))
        if len(results) >= sections: break
    rel = {}
    def add(nid, via, score, heading=None):
        if not nid or nid in prim or nid in rel: return
        rel[nid] = AttrDict(node_id=nid, via=via, score=score or 0.0, breadcrumb=heading)
    if graph and f'{p}entities' in self.t:
        for h in self.graph_search(q, emb, columns=['content', 'heading', 'node_id'],
                                   limit=related * 2, table_name=store, prefix=prefix, graph_w=graph_w):
            add(h.get('node_id'), 'graph', h.get('_rrf_score'), h.get('heading'))
    if vector:
        for h in self.doc_search(q,emb,columns=['content'],limit=related * 2,store=store,prefix=prefix,spans=False):
            add(h.get('node_id'), 'vector', h.get('_rrf_score'), h.get('heading') or h.get('breadcrumb'))
    related_list, seen_rel = L(), set(seen)
    for r in sorted(rel.values(), key=lambda r: -r.score):
        bc = r.breadcrumb or self.breadcrumb(r.node_id, store, prefix)
        if not _prov(bc) or bc in seen_rel: continue
        seen_rel.add(bc); r.breadcrumb = bc
        nd = first(N(where=f'id={r.node_id!r}'))
        r.title = nd['title'] if nd else r.node_id
        r.doc_id, r.filename = doc_of(r.node_id)
        related_list.append(r)
        if len(related_list) >= related: break
    return AttrDict(query=q, results=results, related=related_list)

## End to end

A three-chapter document, a deterministic hash embedder (so the notebook runs offline), and every
call in the module.

In [ ]:
import numpy as np, hashlib
from litesearch.core import database
from litesearch.graph import hash_embed

DOC = """# Saturn

Saturn is the slowest of the classical grahas.

## Sade Sati

Sade sati is the seven and a half year period when Saturn transits the twelfth, first and second
houses from the natal moon. It is counted in three phases of roughly thirty months each.

The middle phase, with Saturn over the moon itself, is held to be the heaviest.

## Remedies

Recitation on Saturdays is the common prescription. Donation of black sesame is another.

# Jupiter

Jupiter is the great benefic and moves through one sign each year.

## Transits

Jupiter's return to its natal sign happens roughly every twelve years."""

db  = database()
enc = lambda ts, **kw: hash_embed(ts, 256)
db.get_tree('store')
print(db.add_doc(DOC, title='Grahas', source='notes/grahas.md', kind='md', emb_fn=enc))

{'doc_id': '5a991f283775ae0d', 'title': 'Grahas', 'kind': 'md', 'nodes': 6, 'chunks': 5}


In [ ]:
# the tree came out of the headings, and every chunk knows which node it lives in
t = db.toc('Grahas')[0]
def show(n, d=0):
    print('  '*d + f"{n['title']:<14} chunks={n['nchunks']}  {n['id']}")
    for c in n.get('children', []): show(c, d+1)
show(t['tree'])
assert [c['title'] for c in t['tree']['children']] == ['Saturn', 'Jupiter']
assert db.breadcrumb(t['tree']['children'][0]['children'][0]['id']) == 'Grahas › Saturn › Sade Sati'

Grahas         chunks=0  5a991f283775ae0d#0
  Saturn         chunks=1  5a991f283775ae0d#1
    Sade Sati      chunks=1  5a991f283775ae0d#2
    Remedies       chunks=1  5a991f283775ae0d#3
  Jupiter        chunks=1  5a991f283775ae0d#4
    Transits       chunks=1  5a991f283775ae0d#5


In [ ]:
# read() returns the section, not a fragment -- and includes its children
sec = first(t['tree']['children'][0]['children'], lambda c: c['title'] == 'Sade Sati')
r = db.read(sec['id'])
print(r['breadcrumb'], '|', r['pages'], '|', len(r['text']), 'chars')
assert 'thirty months' in r['text'] and r['breadcrumb'].endswith('Sade Sati')

Grahas › Saturn › Sade Sati | (0, 0) | 266 chars


In [ ]:
# hybrid search, but every hit is placed in the document
q  = 'how long does the saturn transit last'
hits = db.doc_search(q, enc([q])[0].tobytes(), limit=3, dtype=np.float16)
for h in hits: print(f"{h['_rrf_score']:.4f}  {h['breadcrumb']:<28} {h['content'][:60]!r}")
assert hits and all(h['breadcrumb'] for h in hits)

# sections() rolls the same evidence up a level and hands back the next call
secs = db.sections(q, enc([q])[0].tobytes(), limit=2, dtype=np.float16)
for s in secs: print(f"{s['score']:.4f}  {s['breadcrumb']:<28} {s['read']}")
assert secs and secs[0]['read'].startswith('read(')

0.0167  Grahas › Saturn › Sade Sati  'Sade sati is the seven and a half year period when Saturn tr'
0.0164  Grahas › Saturn              'Saturn is the slowest of the classical grahas.'
0.0161  Grahas › Saturn › Remedies   'Recitation on Saturdays is the common prescription. Donation'
0.0167  Grahas › Saturn › Sade Sati  read('5a991f283775ae0d#2')
0.0164  Grahas › Saturn              read('5a991f283775ae0d#1')


In [ ]:
# adaptive fusion: a quoted phrase leans on FTS, an empty FTS leg hands the vote to vectors
assert adaptive_weights('"sade sati"', [1], [1])[0] > adaptive_weights('how long does it last', [1], [1])[0]
assert adaptive_weights('anything', [], [1]) == (0.0, 1.0)
assert adaptive_weights('anything', [1], []) == (1.0, 0.0)

# spans: two hits one page apart in the same node become one passage
_h = [dict(content='first half', node_id='d#1', page=3, _rrf_score=0.02, rowid=1),
      dict(content='second half', node_id='d#1', page=4, _rrf_score=0.01, rowid=2),
      dict(content='elsewhere',   node_id='d#9', page=40, _rrf_score=0.005, rowid=3)]
_m = merge_spans(_h)
assert len(_m) == 2 and _m[0]['_nspan'] == 2 and 'second half' in _m[0]['content'], _m
assert _m[0]['_rrf_score'] == 0.02, 'a merged span keeps its best score, it does not accumulate'

In [ ]:
# ingestion is content-addressed, and delete_doc leaves nothing behind
again = db.add_doc(DOC, title='Grahas', source='notes/grahas.md', emb_fn=enc)
assert 'skipped' in again, again
did = t['doc_id']
n_before = len(db.t.store())
db.delete_doc(did)
assert len(db.t.store()) == 0 and db.toc() == [] and n_before > 0
print(f'{n_before} chunks removed with the document')

5 chunks removed with the document


## What this does not do yet

- **No LLM anywhere.** `summarize=` takes a callable and `build_tree` can be replaced wholesale;
  an LLM-written summary per node is the single highest-value place to spend tokens, because it is
  what `toc()` shows an agent that is deciding where to read.
- **Chunking is `chunk_markdown`.** A cost-model splitter (sentences → chunklets → semantic merge,
  as in RAGLite) produces better boundaries on prose. The seam is `chunker=`.
- **Late chunking is available but not wired in here.** `litesearch.utils.LateChunkFastEncode`
  contextualises a node's chunks jointly; passing one as `emb_fn` already works, but nothing yet
  batches per node to make it worthwhile.
- **Chunking is where the measured gains are.** On the evaluation in `nbs/07_doc_eval.ipynb`,
  moving from page-sized chunks to node-scoped ones lifted MRR from 0.330 to 0.522 on degraded
  queries — more than every other feature here combined. The cost-model splitter is the next
  thing to port, not a refinement of the ranking.

In [ ]:
import tempfile
_d = Path(tempfile.mkdtemp())
(_d/'ratios.md').write_text('# Ratios\n\n## Liquidity\n\nCurrent ratio is current assets over current liabilities.')
(_d/'notes.txt').write_text('A plain text file with no headings at all, ingested as one windowed node.')
_fdb = database()
_fdb.get_tree('store')
print(_fdb.add_dir(_d, emb_fn=enc))
assert {d['title'] for d in _fdb.toc()} == {'ratios', 'notes'}
_rt = _fdb.toc('ratios')[0]['tree']
assert [n['title'] for n in _rt['children']] == ['Ratios']            # the `# Ratios` heading
assert [n['title'] for n in _rt['children'][0]['children']] == ['Liquidity']
assert _fdb.add_file(_d/'ratios.md', emb_fn=enc).get('skipped'), 're-ingest is a no-op'
assert 'current liabilities' in _fdb.read(_fdb.toc('ratios')[0]['tree']['id'])['text']

[{'doc_id': 'acdecfddcad969a0', 'title': 'notes', 'kind': 'txt', 'nodes': 2, 'chunks': 1}, {'doc_id': 'fca03a0aa3f13fff', 'title': 'ratios', 'kind': 'md', 'nodes': 3, 'chunks': 1}]


In [ ]:
#| hide
# regression: identical text in two sections must be stored under both nodes, not collapsed by a
# content-only hash id. add_doc with emb_fn=None is model-free, so this runs anywhere.
_shared = ("This exact paragraph appears in two different sections and must be stored under both, "
           "not deduped away by a content-only hash id, long enough to survive the min-chunk merge.")
_ddb = database(sem_search=False)
_ddb.add_doc([(0, f"# Alpha\n\n{_shared}\n\n# Beta\n\n{_shared}\n")], title='DupSections')
_rows = list(_ddb.t['store']())
assert len(_rows) == 2, f'both chunks must be stored; got {len(_rows)}'
_leaves = [n['id'] for n in _ddb.t['nodes']() if n['level'] > 0]
assert all(_ddb.read(nid, children=False)['text'].strip() for nid in _leaves), 'a section read back empty -> chunk lost'
assert sum(n['nchunks'] for n in _ddb.t['nodes']()) == len(_rows), 'nchunks must match stored rows'


In [ ]:
#| hide
# a graph built over a doc/tree store must reference the store's real (composite) chunk ids, so
# graph_search can join mentions back to chunks. Feed store rows *with* their id.
from litesearch.graph import build_graph
_gdb = database()
_gdb.add_doc([(0, "# Rights\n\nThe supplier shall inform the consumer of the right of withdrawal.\n\n"
                 "# Duties\n\nThe consumer may exercise the right of withdrawal within the stated period.")],
             title='GraphDoc', emb_fn=lambda ts, **kw: hash_embed(ts, 256))
_grows = list(_gdb.t['store'](select='id, content'))
build_graph(_gdb, _grows, emb_fn=lambda ts, **kw: hash_embed(ts, 256))
_ids = {r['id'] for r in _grows}
_cids = {m['chunk_id'] for m in _gdb.get_graph('store').mentions(select='chunk_id')}
assert _cids and _cids <= _ids, 'graph mentions must reference real store chunk ids'


In [ ]:
#| hide
# context(): one composed retrieval -- operative sections in tree context + related provisions.
import numpy as np
from litesearch.core import database
from litesearch.graph import hash_embed, build_graph
from fastcore.all import L
_cdb = database()
_cenc = lambda ts, **kw: hash_embed(ts, 256)
_cdb.add_doc([(0, "# Rights\n\n## Article 1\n\nThe consumer has a right of withdrawal within fourteen days, "
                 "subject to Article 2 and the definitions therein.\n\n## Article 2\n\nDefinitions: a consumer "
                 "means a natural person acting outside their trade.\n\n# Duties\n\n## Article 3\n\nThe supplier "
                 "shall inform the consumer of the right of withdrawal before the contract is concluded.")],
             title='ConsumerRules', emb_fn=_cenc)
_cemb = hash_embed(['right of withdrawal'], 256)[0].astype(np.float16).tobytes()
_ctx = _cdb.context('right of withdrawal', _cemb, sections=3, related=5)
assert _ctx.results, 'context returned no operative sections'
assert all('›' in r.breadcrumb for r in _ctx.results), 'each result is a provision with a breadcrumb path'
assert _ctx.results[0].text and _ctx.results[0].tree is not None, 'a result carries full text and tree context'
assert isinstance(_ctx.related, L)
build_graph(_cdb, list(_cdb.t['store'](select='id, content')), emb_fn=_cenc)
# graph= is opt-in, so the graph leg needs asking for by name; without this the default
# path stopped covering it at all
assert _cdb.context('right of withdrawal', _cemb, sections=3, related=5, graph=True).results
assert _cdb.context('right of withdrawal', _cemb, sections=3, related=5).results
assert not any(r.via == 'graph' for r in _cdb.context('right of withdrawal', _cemb, related=5).related)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()